<h1 style="text-align: center;">Supports pour le projet </h1>


Ce notebook contient différents éléments pour vous aider à réaliser le projet : récupération des données, exemples de code, fonctions utiles et quelques conseils.

Il ne constitue pas une solution complète du projet. Certaines parties sont volontairement à compléter ou à adapter. Prenez également le temps de consulter les guides indiqués dans le sujet, qui contiennent les notions et fonctions dont vous aurez besoin.


# A FAIRE EN TOUT PREMIER

L'utilisation des modèles Hugging Face avec TensorFlow sur Google Colab peut poser des problèmes de compatibilité entre les différentes versions de TensorFlow, Keras et Transformers.

Si vous utilisez Google Colab, exécutez la cellule suivante avant toute autre cellule du notebook.

Une fois l'installation terminée, redémarrez impérativement l'environnement d'exécution, puis reprenez le notebook à partir de la section suivante.

In [ ]:
# Nettoyage des packages Colab inutiles pour ce TP
%pip -q uninstall -y \
    gradio diffusers \
    keras-hub keras-nlp \
    tensorflow-text tensorflow-decision-forests \
    tf-keras keras

# Versions utilisées dans ce TP
%pip -q install \
    "tensorflow==2.20.0" \
    "tf-keras==2.20.1" \
    "transformers==4.55.4" \
    "safetensors"

# Initialisation pour Colab


Nous allons travailler sous Google Colab. Montez votre Google Drive afin de pouvoir sauvegarder les données, les modèles et les résultats de vos expérimentations.


In [ ]:
# N'oubliez pas d'adapter le répertoire de travail à votre configuration !
from google.colab import drive
drive.mount("/content/gdrive")

my_local_drive = "/content/gdrive/My Drive/Colab Notebooks/ML_FDS"
%cd "$my_local_drive"

Les librairies suivantes seront utilisées dans le projet.


In [1]:
import os
import io
import math
import random
import zipfile
import requests
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Utilisation de Keras 2 avec Transformers
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tf_keras as keras

from tf_keras import layers, Model
from tf_keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tf_keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tf_keras.callbacks import EarlyStopping, ModelCheckpoint
from tf_keras.optimizers import Adam
from tf_keras.utils import register_keras_serializable

from sklearn.preprocessing import LabelEncoder

from transformers import DistilBertTokenizerFast, TFDistilBertModel

# Pour utiliser au mieux le GPU
AUTOTUNE = tf.data.AUTOTUNE

I0000 00:00:1794566380.457549  224706 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794566380.460074  224706 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794566380.767409  224706 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794566382.812786  224706 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

ModuleNotFoundError: No module named 'tf_keras'

# Récupération du jeu de données



Nous commençons avec le jeu de données contenant les quatre classes :

* `bike`
* `ball`
* `water`
* `dog`

Chaque classe contient 150 images avec les textes associés.

Après extraction, le jeu de données est organisé de la manière suivante :

```text
flickr_subset2
├── captions.csv
├── captions
│   ├── bike100.txt
│   ├── bike200.txt
│   └── ...
└── images
    ├── bike
    ├── ball
    ├── water
    └── dog
```

Le fichier `captions.csv` permet notamment de faire la correspondance entre les images, les textes associés et les classes.




In [ ]:
url = "https://www.lirmm.fr/~poncelet/Ressources/flickr_subset2.zip"
target_dir = "flickr_subset2"

if os.path.exists(target_dir) and os.path.isdir(target_dir):
    print("Données déjà disponibles dans :", target_dir)

else:
    print("Téléchargement du jeu de données...")
    response = requests.get(url)

    if response.status_code == 200:
        print("Téléchargement réussi. Extraction...")

        with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
            for member in zip_ref.namelist():

                # Évite de recréer un sous-répertoire flickr_subset2/flickr_subset2
                member_path = member
                if member.startswith("flickr_subset2/"):
                    member_path = member[len("flickr_subset2/"):]

                target_path = os.path.join(target_dir, member_path)

                if member.endswith("/"):
                    os.makedirs(target_path, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target_path), exist_ok=True)

                    with zip_ref.open(member) as source, \
                         open(target_path, "wb") as target:
                        target.write(source.read())

        print("Données extraites dans :", target_dir)

    else:
        print("Échec du téléchargement. Code HTTP :", response.status_code)

# Fonctions, classes et variables utiles pour la suite



Les cellules suivantes contiennent quelques fonctions et classes qui seront utiles dans le projet.

**Attention :** prenez le temps de comprendre leur fonctionnement. Elles seront réutilisées dans plusieurs parties du projet et notamment lors de la construction du modèle CLIP.

**Attention :** il est important que vous compreniez bien le fonctionnement des classes, aussi n'hésitez pas à vous reporter à [Ressources Deep Learning](https://gite.lirmm.fr/poncelet/deeplearning) où vous de plus amples informations sur leur fonctionnement et leur usage.



In [ ]:
# FONCTIONS UTILES

def preview_images_from_dir(image_dir, image_size=(224, 224), n=12,
                            subset="training", seed=123):
    """
    Affiche n images avec leurs labels à partir d'un dossier
    (flow_from_directory).
    N'utilise PAS train_gen pour ne pas en avancer l'index.
    """
    # Générateur léger juste pour l'aperçu
    # Ici il y a une bonne indication si vous regardez bien pour la classif :)
    preview_datagen = ImageDataGenerator(validation_split=0.2, rescale=1./255)

    preview_gen = preview_datagen.flow_from_directory(
        image_dir,
        target_size=image_size,
        batch_size=n,
        class_mode="categorical",
        subset=subset,
        shuffle=True,
        seed=seed
    )

    imgs, y = next(preview_gen)    # imgs: (n, H, W, 3), y: one-hot
    class_names = list(preview_gen.class_indices.keys())
    labels = np.argmax(y, axis=1)

    rows = 3
    cols = int(np.ceil(n / rows))
    plt.figure(figsize=(4*cols, 4*rows))
    for i in range(min(n, imgs.shape[0])):
        plt.subplot(rows, cols, i+1)
        plt.imshow(imgs[i])
        plt.title(class_names[labels[i]])
        plt.axis("off")
    plt.tight_layout()
    plt.show()


Classes utiles pour la suite.

In [ ]:
# Classes utiles pour la suite

@register_keras_serializable()
class SelectFirstToken(layers.Layer):
    """
    Récupère la représentation du premier token de la séquence.
    """
    def call(self, inputs):
        return inputs[:, 0, :]


@register_keras_serializable()
class L2Normalize(layers.Layer):
    """
    Normalise les embeddings avec une norme L2.
    """
    def __init__(self, axis=-1, **kwargs):
        super().__init__(**kwargs)
        self.axis = axis

    def call(self, inputs):
        return tf.math.l2_normalize(inputs, axis=self.axis)

    def get_config(self):
        config = super().get_config()
        config.update({"axis": self.axis})
        return config

Quelques variables utilisées dans la suite du projet sont définies ci-dessous. Elles correspondent aux valeurs utilisées dans nos exemples et devraient convenir pour vos premiers essais.

**Attention :** il est conseillé de conserver les noms de ces variables, car certains codes fournis par la suite les utiliseront directement.


In [ ]:
# Répertoire pour sauvegarder les modèles
model_dir = "./models_forclip"
os.makedirs(model_dir, exist_ok=True)

# Répertoire des données
dataset_dir = "./flickr_subset2"

# Répertoire des images
image_dir = os.path.join(dataset_dir, "images")

# Répertoire des captions
captions_dir = os.path.join(dataset_dir, "captions")

# Classes
# Attention : conserver l'ordre alphabétique utilisé par le générateur
class_names = ["ball", "bike", "dog", "water"]

# Images
image_size = (224, 224)
image_shape = image_size + (3,)

# Textes
text_model_name = "distilbert-base-uncased"
max_len = 32

# Dimension commune des embeddings image et texte pour CLIP
embed_dim = 128

# Partie classification d'images



Nous allons commencer par construire un modèle capable de classifier les images dans les quatre classes `bike`, `ball`, `water` et `dog`.

L'objectif n'est pas de chercher le meilleur classifieur possible, mais d'obtenir un modèle qui fonctionne correctement et que nous pourrons réutiliser ensuite pour construire la partie image du modèle CLIP.

Commençons par afficher quelques images du jeu de données afin de voir ce que nous allons manipuler.

**Remarque :** regardez également la fonction utilisée pour afficher ces images. Elle contient quelques indications qui pourront vous être utiles pour construire votre classifieur.


In [ ]:
# Affichage des images
preview_images_from_dir(
    image_dir,
    image_size=image_size,
    n=12,
    subset="training",
    seed=42
)

**Quelques indications pour la suite :**

* Les images ne doivent pas nécessairement être toutes chargées en mémoire en même temps. Vous pouvez les lire directement à partir des répertoires et les fournir au modèle par batches. La fonction utilisée juste au-dessus pour afficher les images vous donne plusieurs indications sur la manière de faire.

* Pour charger les images directement depuis les répertoires et les fournir au modèle par batches, vous pouvez notamment utiliser `ImageDataGenerator` et la méthode `flow_from_directory` de Keras. Ces fonctions n'ont pas été utilisées dans les TP précédents : prenez le temps de regarder leur fonctionnement dans les ressources disponibles sur https://gite.lirmm.fr/poncelet/deeplearning/.

* Pensez à séparer les données utilisées pour l'apprentissage de celles utilisées pour la validation.

* Les valeurs des pixels devront être mises à une échelle adaptée avant d'être fournies au réseau.

* Pour ce problème de classification multiclasse, vous pourrez utiliser `categorical_crossentropy` comme fonction de coût et `adam` comme optimiseur.

* L'entraînement pouvant prendre un certain temps, pensez à sauvegarder automatiquement le meilleur modèle au cours de l'apprentissage. Le callback `ModelCheckpoint` de Keras permet notamment de le faire.

**Conseil :** avant de lancer un apprentissage long, vérifiez toujours que votre modèle est correctement sauvegardé et que vous êtes capable de le recharger.





In [ ]:
# A FAIRE




# Partie classification de textes

Nous allons maintenant construire un modèle capable de classifier les textes associés aux images dans les mêmes quatre classes : `bike`, `ball`, `water` et `dog`.

Pour cela, nous utiliserons DistilBERT, que vous avez déjà manipulé dans le TP précédent, ainsi que son tokenizer.

**Quelques indications pour la suite :**

* Les textes et leurs labels sont disponibles dans le fichier `captions.csv`.

* Utilisez le tokenizer associé à DistilBERT pour transformer les textes dans le format attendu par le modèle. Pensez notamment à la gestion de la longueur des séquences, du padding et de la troncature.

* DistilBERT produit une représentation pour chacun des tokens de la séquence. Sa sortie est donc de la forme `(batch, sequence_length, hidden_size)`. Pour réaliser une classification du texte complet, il faut choisir comment obtenir une représentation unique de chaque phrase.

* À partir de cette représentation, vous pourrez ajouter une ou plusieurs couches permettant de réaliser la classification dans les quatre classes.

Comme pour les images, l'objectif n'est pas de chercher le meilleur classifieur possible, mais d'obtenir un modèle qui fonctionne correctement et que nous pourrons ensuite réutiliser pour construire la partie texte du modèle CLIP.



In [ ]:
# A FAIRE

# Partie CLIP

## Construction du modèle

Nous avons maintenant construit deux classifieurs : un pour les images et un pour les textes. Pour construire CLIP, nous n'avons plus besoin de prédire une classe. Nous allons plutôt utiliser les représentations produites par les deux modèles pour construire un encodeur d'images et un encodeur de textes.

CLIP apprend à placer une image et le texte qui lui correspond à proximité dans un espace commun. Les deux encodeurs doivent donc produire des embeddings de même dimension.

Après leur projection dans cet espace commun, les embeddings image et texte doivent être normalisés avec une norme L2. Leur norme devient alors égale à 1 et leur produit scalaire correspond à leur similarité cosinus.

L'objectif du modèle CLIP sera d'obtenir une similarité élevée pour les couples image-texte correspondants et une similarité faible pour les autres couples du batch.

<center>
    <img src="https://www.lirmm.fr/~poncelet/Ressources/architectureCLIP.png" width="480" height="400">
    <p>Le principe du fonctionnement d'un CLIP (source : https://viso.ai/deep-learning/clip-machine-learning/)</p>
</center>



**Astuce :** avant de construire le modèle CLIP, posez-vous les bonnes questions :

* combien d'entrées possède le modèle ?
* quelles sont les entrées de chaque encodeur ?
* quelles doivent être leurs sorties ?
* quelles dimensions doivent être communes aux deux encodeurs ?

Essayez de bien identifier, pour chaque composante, les formes des tenseurs en entrée et en sortie. N'hésitez pas à faire un petit schéma de vos encodeurs pour vérifier que tout est cohérent avant de construire le modèle.

Pour l'encodeur d'images, l'entrée est de la forme `(B, H, W, C)` où `B` est la taille du batch, `H` et `W` les dimensions de l'image et `C` le nombre de canaux. Sa sortie doit être un embedding de forme `(B, D)`.

Pour l'encodeur de textes, DistilBERT reçoit deux entrées produites par le tokenizer :

* `input_ids` de forme `(B, L)` ;
* `attention_mask` de forme `(B, L)`.

`L` correspond à la longueur maximale des séquences. L'encodeur de textes doit lui aussi produire un embedding de forme `(B, D)`.

Dans notre cas, nous pouvons par exemple avoir `H=224`, `W=224`, `C=3` et `D=128`.

Ainsi, pour un batch de taille `B`, le modèle CLIP reçoit :

* des images : `(B, 224, 224, 3)` ;
* des `input_ids` : `(B, L)` ;
* des `attention_mask` : `(B, L)`.

Et il produit :

* des embeddings image : `(B, D)` ;
* des embeddings texte : `(B, D)`.

Les deux embeddings doivent avoir la même dimension `D` et être normalisés avant le calcul des similarités.



## A propos de la loss



Pour entraîner CLIP, nous allons utiliser une loss contrastive. Son objectif est de rapprocher les représentations des images et des textes qui correspondent, tout en éloignant celles qui ne correspondent pas.

Considérons un batch contenant `B` couples `(image, texte)`. Après passage dans les deux encodeurs, nous disposons de :

* `B` embeddings d'images de forme `(B, D)` ;
* `B` embeddings de textes de forme `(B, D)`.

Les embeddings ayant été normalisés, nous pouvons calculer toutes les similarités image-texte du batch par un produit matriciel. Nous obtenons ainsi une matrice de taille `B × B` :

$$
logits = \frac{image\_embeds \times text\_embeds^T}{\tau}
$$

où `τ` est la température.

La case `(i,j)` contient la similarité entre l'image `i` et le texte `j`. Comme le batch est construit de façon à ce que l'image `i` corresponde au texte `i`, les bonnes associations se trouvent sur la diagonale.

Les labels peuvent donc simplement être :

```python
labels = [0, 1, 2, ..., B-1]
```

La loss est calculée dans les deux directions :

* image → texte : chaque image doit retrouver son texte ;
* texte → image : chaque texte doit retrouver son image.

On calcule les deux cross-entropies puis on en prend la moyenne.

Pour vous faciliter cette partie, la couche `ClipLossLayer` ci-dessous vous est fournie. Elle reçoit simultanément les embeddings image et texte, calcule la loss contrastive et l'ajoute directement au modèle avec `add_loss()`.



La Figure suivante illustre un exemple de matrice et de rapprochement.


<center>
    <img src="https://www.lirmm.fr/~poncelet/Ressources/exemplematriceclip.png" width="600" height="500">
    <p>Un exemple de matrice (source : https://openai.com/index/clip/)</p>
</center>

In [ ]:
# ============================================================
# Loss contrastive utilisée pour entraîner CLIP
# ============================================================
#
# Cette couche reçoit deux matrices :
#   img : embeddings des images  -> shape (B, D)
#   txt : embeddings des textes  -> shape (B, D)
#
# avec :
#   B = taille du batch
#   D = dimension commune des embeddings
#
# Les embeddings doivent déjà être normalisés avec une norme L2.
# Leur produit scalaire correspond alors à une similarité cosinus.
#
# Dans le batch, on suppose que :
#   image 0 <-> texte 0
#   image 1 <-> texte 1
#   ...
#   image i <-> texte i
#
# Les bonnes associations se trouvent donc sur la diagonale de la
# matrice de similarité image / texte.

@register_keras_serializable(package="clip")
class ClipLossLayer(layers.Layer):

    def __init__(self, temperature=0.07, **kwargs):
        super().__init__(**kwargs)

        # La température contrôle la "netteté" du softmax.
        # Plus elle est petite, plus le modèle doit distinguer fortement
        # les bonnes associations des mauvaises.
        self.temperature = temperature

    def call(self, inputs):

        # Récupération des deux ensembles d'embeddings.
        #
        # img : (B, D)
        # txt : (B, D)
        img, txt = inputs

        # ------------------------------------------------------------
        # 1. Calcul de toutes les similarités image / texte du batch
        # ------------------------------------------------------------
        #
        # txt^T a pour forme (D, B)
        #
        #       (B, D) x (D, B) -> (B, B)
        #
        # La case logits[i, j] contient donc la similarité entre :
        #   l'image i
        #   et le texte j
        #
        # Comme les embeddings sont L2-normalisés, le produit scalaire
        # correspond à une similarité cosinus.
        #
        # On divise ensuite par la température.

        logits = tf.matmul(
            img,
            txt,
            transpose_b=True
        ) / self.temperature

        # logits a donc une forme (B, B).
        #
        # Exemple pour B = 4 :
        #
        #                texte
        #             0   1   2   3
        # image 0     x   .   .   .
        # image 1     .   x   .   .
        # image 2     .   .   x   .
        # image 3     .   .   .   x
        #
        # Les x de la diagonale correspondent aux bonnes associations.

        # ------------------------------------------------------------
        # 2. Construction des labels
        # ------------------------------------------------------------
        #
        # Pour l'image 0, le bon texte est le texte 0.
        # Pour l'image 1, le bon texte est le texte 1.
        # ...
        #
        # Les labels sont donc simplement :
        # [0, 1, 2, ..., B-1]

        labels = tf.range(tf.shape(logits)[0])

        # ------------------------------------------------------------
        # 3. Loss image -> texte
        # ------------------------------------------------------------
        #
        # Chaque ligne de logits correspond à une image.
        #
        # On demande donc :
        # "parmi tous les textes du batch, lequel correspond
        #  à cette image ?"
        #
        # Pour la ligne i, la bonne réponse est la colonne i.

        loss_i = keras.losses.sparse_categorical_crossentropy(
            labels,
            logits,
            from_logits=True
        )

        # ------------------------------------------------------------
        # 4. Loss texte -> image
        # ------------------------------------------------------------
        #
        # On veut aussi faire le raisonnement inverse :
        #
        # "parmi toutes les images du batch, laquelle correspond
        #  à ce texte ?"
        #
        # Il suffit pour cela de transposer la matrice de similarité.

        loss_t = keras.losses.sparse_categorical_crossentropy(
            labels,
            tf.transpose(logits),
            from_logits=True
        )

        # ------------------------------------------------------------
        # 5. Loss CLIP finale
        # ------------------------------------------------------------
        #
        # On moyenne les deux directions :
        #
        #   image -> texte
        #   texte -> image
        #
        # puis on moyenne sur l'ensemble du batch.

        loss = tf.reduce_mean(loss_i + loss_t) / 2.0

        # ------------------------------------------------------------
        # 6. Ajout de la loss au modèle
        # ------------------------------------------------------------
        #
        # Cette couche est directement insérée dans le modèle CLIP.
        # add_loss() permet donc d'attacher cette loss au graphe Keras.
        #
        # Grâce à cela, nous n'aurons pas besoin de fournir une loss
        # classique dans model.compile().

        self.add_loss(loss)

        # On ne transforme pas les embeddings :
        # ils sont simplement renvoyés pour pouvoir continuer
        # à les utiliser, notamment pour l'inférence et la recherche.

        return (img, txt)

    def get_config(self):
        # Permet de sauvegarder puis recharger correctement la couche
        # avec la valeur de la température.
        config = super().get_config()
        config.update({"temperature": self.temperature})
        return config


Nous avons maintenant une couche capable de calculer la loss contrastive à partir des embeddings image et texte.

Il reste cependant une particularité importante à comprendre. Notre modèle CLIP produit deux sorties :

* les embeddings des images ;
* les embeddings des textes.

Or la loss contrastive doit utiliser **les deux simultanément** pour construire la matrice de similarité `B × B`.

On ne peut donc pas simplement écrire :

```python
model.compile(loss=...)
```

comme pour un problème de classification classique. Avec plusieurs sorties, Keras associe normalement une loss à chaque sortie séparément. Ici ce n'est pas ce que nous voulons : notre loss doit comparer directement les embeddings image et texte.

C'est pour cette raison que `ClipLossLayer` est directement insérée dans le modèle à l'endroit où les deux embeddings sont disponibles :

```text
image ──> encodeur image ──> embedding image ──┐
                                               ├──> ClipLossLayer
texte ──> encodeur texte ──> embedding texte ──┘
```

La méthode `add_loss()` utilisée dans cette couche permet d'ajouter la loss contrastive directement au modèle.

Ainsi, lors de la compilation du modèle, nous aurons uniquement besoin de préciser l'optimiseur : la loss est déjà définie à l'intérieur du modèle.


In [ ]:
# A FAIRE

# A propos du jeu de données d'apprentissage


Pour entraîner CLIP, chaque image doit être associée au texte qui lui correspond. Cette correspondance est particulièrement importante puisque, dans chaque batch, la loss considère que l'image `i` est associée au texte `i`.

Notre modèle reçoit maintenant trois tenseurs :

* les images, de forme `(B, H, W, C)` ;
* les `input_ids` produits par le tokenizer de DistilBERT, de forme `(B, L)` ;
* les `attention_mask`, également de forme `(B, L)`.

Le fichier `captions.csv` contient les informations nécessaires pour retrouver chaque image et le texte qui lui est associé.

Pour éviter de passer trop de temps sur la préparation des batches, une fonction vous est fournie ci-dessous. Elle construit un `tf.data.Dataset` à partir de `captions.csv`. Pour chaque exemple, elle :

* charge l'image correspondante ;
* la redimensionne en `224 × 224` et normalise ses valeurs ;
* tokenize le texte avec le tokenizer de DistilBERT ;
* retourne ensemble l'image, les `input_ids` et l'`attention_mask`.

La fonction utilise également `map(..., num_parallel_calls=AUTOTUNE)` et `prefetch(AUTOTUNE)` afin de préparer les données efficacement pendant l'entraînement.

**Attention :** l'ordre entre les images et les textes ne doit jamais être modifié indépendamment. Si l'image située en position `i` ne correspond plus au texte situé en position `i`, la diagonale de la matrice de similarité ne représentera plus les bonnes associations.


In [ ]:
# ============================================================
# Construction d'un tf.data.Dataset pour entraîner CLIP
# ============================================================
#
# Chaque exemple du dataset CLIP doit contenir :
#
#   - une image
#   - les input_ids du texte correspondant
#   - l'attention_mask du texte correspondant
#
# IMPORTANT :
# l'image située à la position i doit toujours correspondre
# au texte situé à la position i.
#
# La loss CLIP considérera en effet que les bonnes associations
# se trouvent sur la diagonale de la matrice de similarité.


def make_clip_dataset(
    dataframe,
    tokenizer,
    dataset_root,
    batch_size=32,
    max_len=32,
    image_size=(224, 224),
    shuffle=True,
    seed=42
):
    """
    Construit un tf.data.Dataset utilisable par notre modèle CLIP.

    Paramètres
    ----------
    dataframe :
        DataFrame contenant au minimum les colonnes :
            - image_path
            - caption

    tokenizer :
        tokenizer associé à DistilBERT.

    dataset_root :
        répertoire racine du jeu de données
        (par exemple "./flickr_subset2").

    batch_size :
        nombre de couples image/texte par batch.

    max_len :
        longueur maximale des séquences de tokens.

    image_size :
        taille utilisée pour redimensionner les images.

    shuffle :
        True pour mélanger les couples image/texte.

    Retour
    ------
    Un tf.data.Dataset dont chaque élément est un dictionnaire :

        {
            "image": image,
            "input_ids": input_ids,
            "attention_mask": attention_mask
        }

    Les shapes après création des batches seront :

        image          : (B, H, W, 3)
        input_ids      : (B, L)
        attention_mask : (B, L)
    """


    # ========================================================
    # 1. Récupération des chemins des images et des captions
    # ========================================================

    # Dans captions.csv, image_path contient un chemin relatif,
    # par exemple :
    #
    # images/dog/dog001.jpg
    #
    # On construit donc le chemin complet à partir de dataset_root.

    image_paths = [
        str(Path(dataset_root) / p)
        for p in dataframe["image_path"].astype(str)
    ]

    # On récupère les textes.
    # fillna("") évite un problème éventuel si une caption est vide.
    captions = (
        dataframe["caption"]
        .fillna("")
        .astype(str)
        .tolist()
    )


    # ========================================================
    # 2. Tokenisation de tous les textes
    # ========================================================
    #
    # DistilBERT ne prend pas directement des chaînes de caractères.
    #
    # Le tokenizer transforme chaque texte en deux informations :
    #
    # input_ids :
    #     identifiants des tokens
    #
    # attention_mask :
    #     indique quels tokens doivent réellement être pris en compte
    #     (1 = vrai token, 0 = padding)
    #
    # padding="max_length" permet d'avoir des séquences ayant toutes
    # exactement la même longueur.
    #
    # truncation=True coupe les textes trop longs.

    encoded = tokenizer(
        captions,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="np"
    )

    # Le modèle Keras attend des entiers 32 bits.
    input_ids = encoded["input_ids"].astype("int32")
    attention_masks = encoded["attention_mask"].astype("int32")


    # ========================================================
    # 3. Création du tf.data.Dataset
    # ========================================================
    #
    # Pour l'instant, le dataset contient :
    #
    #   chemin_image
    #   input_ids
    #   attention_mask
    #
    # Attention : ces trois éléments restent toujours ensemble.
    # Si on mélange le dataset, TensorFlow mélange le triplet complet
    # et ne casse donc pas l'association image / texte.

    ds = tf.data.Dataset.from_tensor_slices(
        (
            image_paths,
            input_ids,
            attention_masks
        )
    )


    # ========================================================
    # 4. Mélange éventuel des exemples
    # ========================================================
    #
    # On mélange le dataset d'entraînement pour éviter que les
    # exemples soient toujours présentés dans le même ordre.
    #
    # Pour le jeu de validation, on utilisera shuffle=False.

    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(image_paths),
            seed=seed,
            reshuffle_each_iteration=True
        )


    # ========================================================
    # 5. Chargement et préparation d'un exemple
    # ========================================================

    def load_sample(img_path, input_ids, attention_mask):

        # Lecture du fichier image
        img = tf.io.read_file(img_path)

        # Décodage JPEG en image RGB (3 canaux)
        img = tf.image.decode_jpeg(
            img,
            channels=3
        )

        # Conversion en float32 et normalisation des pixels entre 0 et 1.
        #
        # Une image JPEG contient initialement des valeurs entre 0 et 255.
        img = tf.image.convert_image_dtype(
            img,
            tf.float32
        )

        # Toutes les images doivent avoir la même taille pour pouvoir
        # être regroupées dans un batch.
        img = tf.image.resize(
            img,
            image_size
        )



        # ----------------------------------------------------
        # Entrées du modèle CLIP
        # ----------------------------------------------------
        #
        # Les noms utilisés ici doivent correspondre aux noms
        # des entrées définies dans build_clip_model().
        #
        # On ne retourne PAS de label y :
        # la loss contrastive est directement ajoutée dans
        # ClipLossLayer avec add_loss().

        return {
            "image": img,
            "input_ids": input_ids,
            "attention_mask": attention_mask
        }


    # ========================================================
    # 6. Application du prétraitement aux exemples
    # ========================================================
    #
    # map() applique load_sample à chaque exemple.
    #
    # AUTOTUNE permet à TensorFlow de choisir automatiquement
    # combien d'exemples préparer en parallèle.

    ds = ds.map(
        load_sample,
        num_parallel_calls=AUTOTUNE
    )


    # ========================================================
    # 7. Création des batches
    # ========================================================
    #
    # Chaque batch contient B couples image / texte.
    #
    # Il n'est pas nécessaire d'utiliser drop_remainder=True :
    # si le dernier batch est plus petit, ClipLossLayer construira
    # simplement une matrice de similarité plus petite.

    ds = ds.batch(
        batch_size,
        drop_remainder=False
    )


    # ========================================================
    # 8. Préchargement du batch suivant
    # ========================================================
    #
    # Pendant que le GPU traite un batch, TensorFlow peut déjà
    # préparer le suivant. Cela évite de perdre du temps entre
    # deux batches.

    ds = ds.prefetch(AUTOTUNE)

    return ds

**Conseil :** avant de construire et d'entraîner le modèle CLIP, vérifiez toujours les formes des tenseurs produits par votre dataset. C'est une source fréquente d'erreurs.

Avec un batch de taille `B`, vous devez obtenir trois entrées :

* les images : `(B, 224, 224, 3)` ;
* les `input_ids` : `(B, L)` ;
* les `attention_mask` : `(B, L)`,

où `L` correspond à la longueur maximale utilisée lors de la tokenisation (`max_len`).

La cellule suivante vous permet de construire le dataset puis d'examiner un premier batch.


In [ ]:
# Exemple de création du dataset CLIP
#
# On considère que le tokenizer DistilBERT a déjà été créé précédemment.
# On utilise ici l'ensemble du fichier uniquement pour vérifier
# que la fonction fonctionne correctement et que les formes
# des tenseurs obtenus sont celles attendues.
#
# Il n'y a donc pas encore de séparation apprentissage / validation :
# cette cellule sert uniquement à tester la préparation des données.

df = pd.read_csv(
    os.path.join(dataset_dir, "captions.csv")
)

clip_dataset = make_clip_dataset(
    dataframe=df,
    tokenizer=tokenizer,
    dataset_root=dataset_dir,
    batch_size=32,
    max_len=max_len,
    image_size=image_size,
    shuffle=True
)

# Récupération d'un batch pour vérifier ce qui sera envoyé au modèle
batch = next(iter(clip_dataset))

print("Entrées du batch :", batch.keys())
print("Images         :", batch["image"].shape)
print("Input IDs      :", batch["input_ids"].shape)
print("Attention mask :", batch["attention_mask"].shape)

Vous avez maintenant les éléments nécessaires pour construire et entraîner votre modèle CLIP.

Avant de lancer l'apprentissage, vérifiez cependant que tout est cohérent.

**Questions à se poser :**

* Est-ce que mon tokenizer DistilBERT est correctement initialisé ?
* Est-ce que mes textes sont bien transformés en `input_ids` et `attention_mask` ?
* Est-ce que mon dataset fournit bien les trois entrées attendues par le modèle : `image`, `input_ids` et `attention_mask` ?
* Est-ce que mes encodeurs image et texte produisent des embeddings de même dimension ?
* Est-ce que les embeddings sont bien normalisés avant le calcul de la loss contrastive ?
* Est-ce que j'ai séparé mes données en un jeu d'apprentissage et un jeu de validation ?

Pour l'apprentissage, pensez également à utiliser des callbacks. Au minimum :

* `ModelCheckpoint` pour conserver les meilleurs poids obtenus sur le jeu de validation. Avec un modèle contenant DistilBERT, sauvegardez uniquement les poids avec `save_weights_only=True` (par exemple dans un fichier `*.weights.h5`) ;
* `EarlyStopping` pour arrêter l'apprentissage lorsque la `val_loss` ne s'améliore plus. Il est conseillé d'utiliser `restore_best_weights=True` afin de récupérer automatiquement les meilleurs paramètres obtenus pendant l'apprentissage.

Vous pouvez également utiliser `ReduceLROnPlateau` pour diminuer le learning rate lorsque la `val_loss` stagne.

Rappel : la loss contrastive est déjà intégrée dans la définition du modèle grâce à `ClipLossLayer`. Vous n'avez donc pas besoin de définir une nouvelle loss dans `compile()`. Il faut principalement choisir l'optimiseur et son learning rate.

Enfin, avant de lancer un apprentissage long, commencez par une ou deux epochs. Vérifiez que l'entraînement et la validation fonctionnent, que les poids sont correctement sauvegardés et que vous êtes capables de reconstruire le même modèle puis de recharger ces poids avec `load_weights()`.



In [ ]:
 # A FAIRE



Une fois l'apprentissage terminé, vérifiez que les poids sauvegardés peuvent être correctement rechargés.

Attention : `load_weights()` ne reconstruit pas le modèle. Vous devez d'abord recréer exactement la même architecture que celle utilisée pendant l'apprentissage, puis charger les poids sauvegardés.

La cellule suivante vous donne le principe général. À vous de compléter la reconstruction de vos encodeurs.


In [ ]:
# A COMPLETER
# ============================================================
# Vérification du rechargement des poids
# ============================================================

# IMPORTANT :
# pour recharger des poids, il faut d'abord reconstruire
# exactement la même architecture que celle utilisée
# pendant l'apprentissage.

# À vous de reconstruire vos deux encodeurs
image_encoder_reload = ...
text_encoder_reload = ...

# Puis de reconstruire votre modèle CLIP
clip_reloaded = build_clip_model(
    image_encoder_reload,
    text_encoder_reload,
    temperature=0.07
)

# Chargement des meilleurs poids sauvegardés
clip_reloaded.load_weights(best_path)

print("Poids du modèle CLIP correctement rechargés.")

# Faire de l'inférence



Avec un classifieur classique, l'inférence consiste généralement à appeler `predict(...)` afin d'obtenir directement une classe ou une probabilité.

Avec CLIP, le principe est différent. Le modèle n'a pas appris à prédire directement une classe : il a appris deux encodeurs capables de représenter les images et les textes dans un même espace.

L'objectif de l'apprentissage était de rapprocher les embeddings des couples image-texte correspondants et d'éloigner les autres.

Pour utiliser le modèle après l'apprentissage, nous allons donc exploiter directement ces embeddings.

Par exemple, si un utilisateur fournit une requête textuelle comme :

`"A group of cyclists"`

nous pouvons :

1. calculer l'embedding de cette requête avec l'encodeur de textes ;
2. comparer cet embedding aux embeddings des images disponibles ;
3. récupérer les images ayant les similarités les plus élevées.

On peut également effectuer l'opération inverse : fournir une image et rechercher les textes les plus proches.

Le principe est proche de celui utilisé dans la partie *retrieval* d'un système RAG : les documents sont représentés par des embeddings, puis une requête est également transformée en embedding afin de retrouver les éléments les plus proches.

## Construction d'un index d'embeddings

Avant de réaliser une recherche, nous allons créer une base contenant les embeddings des éléments que nous souhaitons interroger.

Pour les images :

* récupérer toutes les images du jeu de données ;
* appliquer le même prétraitement que pendant l'apprentissage ;
* utiliser l'encodeur d'images pour calculer leur embedding ;
* sauvegarder ces embeddings ainsi que les chemins des images correspondantes.

Nous ferons ensuite la même chose pour les textes contenus dans `captions.csv`.

Cette étape permet de calculer les embeddings une seule fois. Lorsqu'une nouvelle requête sera fournie, il suffira alors de calculer son embedding et de le comparer aux embeddings déjà stockés.



In [ ]:
# A COMPLETER

# ============================================================
# Construction d'un index d'embeddings pour les images
# ============================================================
#
# L'objectif est de calculer UNE FOIS l'embedding de toutes
# les images de notre galerie, puis de les sauvegarder.
#
# Lorsqu'une requête texte sera ensuite fournie, nous n'aurons
# plus besoin de recalculer tous ces embeddings : il suffira
# de calculer celui de la requête et de le comparer à l'index.


# ============================================================
# 1. Récupération de l'encodeur d'images
# ============================================================
#
# Votre modèle CLIP contient notamment l'encodeur d'images
# construit précédemment.
#
# À vous de récupérer cet encodeur afin de pouvoir transformer
# une image en un embedding de dimension D.

image_encoder = ...      # À COMPLETER


# ============================================================
# 2. Liste des images de la galerie
# ============================================================
#
# On récupère tous les fichiers .jpg présents dans les
# différents sous-répertoires du jeu de données.

gallery_paths = sorted(
    [str(p) for p in Path(image_dir).rglob("*.jpg")]
)

print("Nombre d'images dans la galerie :", len(gallery_paths))


# ============================================================
# 3. Prétraitement d'une image
# ============================================================
#
# Le prétraitement doit être le même que celui utilisé
# pendant l'apprentissage :
#
#   - lecture de l'image ;
#   - passage en RGB ;
#   - redimensionnement en 224 x 224 ;
#   - conversion en float32 avec des valeurs entre 0 et 1.

def load_img_224(path):

    # Lecture du fichier
    img = tf.io.read_file(path)

    # Décodage en image RGB
    img = tf.image.decode_jpeg(
        img,
        channels=3
    )

    # Conversion en float32 et normalisation entre 0 et 1
    img = tf.image.convert_image_dtype(
        img,
        tf.float32
    )

    # Toutes les images doivent avoir la taille attendue
    # par l'encodeur.
    img = tf.image.resize(
        img,
        image_size
    )

    return img


# ============================================================
# 4. Création d'un tf.data.Dataset
# ============================================================
#
# Cela permet de calculer les embeddings par batches plutôt
# que de traiter les images une par une.

ds_gallery = (
    tf.data.Dataset
    .from_tensor_slices(gallery_paths)
    .map(load_img_224, num_parallel_calls=AUTOTUNE)
    .batch(64)
    .prefetch(AUTOTUNE)
)


# ============================================================
# 5. Calcul des embeddings
# ============================================================
#
# Si N est le nombre d'images de la galerie et D la dimension
# commune choisie pour CLIP, le résultat doit avoir la forme :
#
#                       (N, D)
#
# Utilisez ici votre encodeur d'images.

gallery_embeds = ...     # À COMPLETER


print("Shape des embeddings :", gallery_embeds.shape)


# ============================================================
# 6. Sauvegarde de l'index
# ============================================================
#
# On sauvegarde :
#
#   embeds : les embeddings des images -> (N, D)
#   paths  : les chemins correspondants -> (N,)
#
# Il est important de conserver les deux ensemble afin de
# pouvoir retrouver ensuite l'image correspondant à chaque
# embedding.

index_path = os.path.join(
    model_dir,
    "image_index.npz"
)

np.savez_compressed(
    index_path,
    embeds=gallery_embeds.astype("float32"),
    paths=np.array(gallery_paths, dtype=np.str_)
)

print("Index enregistré :", index_path)
print("Nombre d'images   :", len(gallery_paths))
print("Shape embeddings  :", gallery_embeds.shape)

La cellule suivante illustre comment récupérer le fichier sauvegardé :    

In [ ]:
# ============================================================
# Rechargement de l'index des embeddings d'images
# ============================================================
#
# L'index contient :
#   - les embeddings des images ;
#   - les chemins des images correspondantes.
#
# L'embedding situé à la position i correspond donc
# à l'image dont le chemin est paths[i].

index_path = os.path.join(
    model_dir,
    "image_index.npz"
)

# Chargement du fichier sauvegardé
data = np.load(index_path)

# Embeddings des images
# Shape attendue : (N, D)
embeds = data["embeds"]

# Chemins des images
# Shape attendue : (N,)
paths = data["paths"].astype(str)

print("Shape des embeddings :", embeds.shape)
print("Nombre d'images       :", len(paths))

# Vérification simple
print("\nExemple :")
print("Chemin :", paths[0])
print("Dimension de l'embedding :", embeds[0].shape)

Le code suivant permet de créer les embeddings du texte.


In [ ]:
# A COMPLETER

# ============================================================
# Construction d'un index d'embeddings pour les textes
# ============================================================
#
# Le principe est le même que pour les images :
#
#   caption -> tokenizer DistilBERT -> encodeur texte -> embedding
#
# On calcule une seule fois les embeddings de toutes les captions
# puis on les sauvegarde.


# ============================================================
# 1. Récupération de l'encodeur de textes
# ============================================================
#
# À vous de récupérer l'encodeur texte construit et entraîné
# dans votre modèle CLIP.

text_encoder = ...      # À COMPLETER


# ============================================================
# 2. Récupération des captions
# ============================================================

captions_csv_path = os.path.join(
    dataset_dir,
    "captions.csv"
)

caps_df = pd.read_csv(captions_csv_path)

captions = (
    caps_df["caption"]
    .fillna("")
    .astype(str)
    .tolist()
)

print("Nombre de captions :", len(captions))


# ============================================================
# 3. Tokenisation avec DistilBERT
# ============================================================
#
# Comme pendant l'apprentissage, DistilBERT a besoin de :
#
#   - input_ids
#   - attention_mask
#
# Chaque séquence est ramenée à la même longueur max_len.

encoded = tokenizer(
    captions,
    padding="max_length",
    truncation=True,
    max_length=max_len,
    return_tensors="np"
)

input_ids = encoded["input_ids"].astype("int32")
attention_masks = encoded["attention_mask"].astype("int32")

print("Shape input_ids      :", input_ids.shape)
print("Shape attention_mask :", attention_masks.shape)


# ============================================================
# 4. Calcul des embeddings des textes
# ============================================================
#
# L'encodeur texte attend les deux entrées :
#
#   input_ids
#   attention_mask
#
# Si N est le nombre de captions et D la dimension de
# l'espace commun CLIP, le résultat doit être :
#
#                       (N, D)

text_embeds = ...      # À COMPLETER

print("Shape des embeddings texte :", text_embeds.shape)


# ============================================================
# 5. Sauvegarde de l'index texte
# ============================================================
#
# Pour chaque embedding, on conserve :
#
#   - la caption correspondante ;
#   - le chemin de l'image associée.
#
# Ainsi :
#
# text_embeds[i]
# captions[i]
# image_paths[i]
#
# correspondent tous au même exemple.

text_index_path = os.path.join(
    model_dir,
    "text_index.npz"
)

np.savez_compressed(
    text_index_path,
    embeds=text_embeds.astype("float32"),
    captions=np.array(captions, dtype=np.str_),
    image_paths=caps_df["image_path"].astype(str).to_numpy(dtype=np.str_)
)

print("Index texte enregistré :", text_index_path)
print("Nombre de captions      :", len(captions))
print("Shape embeddings        :", text_embeds.shape)

Pour recharger les embeddings :

In [ ]:
# ============================================================
# Rechargement de l'index des embeddings de textes
# ============================================================

text_index_path = os.path.join(
    model_dir,
    "text_index.npz"
)

data = np.load(text_index_path)

text_embeds = data["embeds"]
captions = data["captions"].astype(str)
image_paths = data["image_paths"].astype(str)

print("Shape des embeddings :", text_embeds.shape)
print("Nombre de captions    :", len(captions))

print("\nExemple :")
print("Caption :", captions[0])
print("Image   :", image_paths[0])
print("Dimension de l'embedding :", text_embeds[0].shape)

## Interroger les index

Nous disposons maintenant de deux index :

* un index contenant les embeddings des images ;
* un index contenant les embeddings des textes.

Nous pouvons donc utiliser notre modèle CLIP pour effectuer une recherche dans les deux sens :

* **texte → images** : à partir d'une requête textuelle, retrouver les images les plus proches ;
* **image → textes** : à partir d'une image, retrouver les captions les plus proches.

### Recherche d'images à partir d'un texte

Considérons par exemple la requête :

`"A group of cyclists"`

Il faut tout d'abord appliquer exactement le même prétraitement que pendant l'apprentissage : tokenisation avec DistilBERT, puis passage dans l'encodeur de textes. On obtient ainsi un embedding `q` de forme `(1, D)`.

Les embeddings des `N` images de la galerie ont été sauvegardés dans une matrice `embeds` de forme `(N, D)`.

Comme les embeddings ont été normalisés avec une norme L2, leur produit scalaire correspond à la similarité cosinus. On peut donc calculer simultanément la similarité entre la requête et toutes les images avec :

```python
scores = (q @ embeds.T)[0]
```

Les dimensions sont alors :

```text
(1, D) @ (D, N) -> (1, N)
```

`scores[i]` représente donc la similarité entre la requête et l'image `paths[i]`.

Il ne reste plus qu'à trier ces scores par ordre décroissant pour récupérer les `top-k` images les plus proches.

### Recherche de textes à partir d'une image

Le principe est exactement le même dans l'autre sens.

Cette fois, l'image fournie comme requête est transformée en embedding avec l'encodeur d'images, puis cet embedding est comparé à tous les embeddings présents dans l'index texte.

Vous allez maintenant implémenter ces deux fonctions de recherche.




In [ ]:
# A FAIRE